# Phase 2: Model Training
**Project:** Anomaly Detection in Network Traffic (UNSW-NB15)

**Goal of this phase:** use the cleaned data from Phase 1 to build a model that can tell **normal traffic (label 0)** from **attack traffic (label 1)**, tune it to the best performance we can get, and save it so Phase 3 (deployment) can use it.

**What we will do, in order**
1. Install and import libraries
2. Load the cleaned data from Phase 1
3. Split a validation set out of the training data
4. Define how we measure a model (metrics)
5. Train 4 baseline models and compare them
6. Try one unsupervised anomaly detector (Isolation Forest)
7. Tune the best model
8. Test the final model on the test set (only once)
9. See which attack types we catch and which we miss
10. See which features matter most
11. Save the model and everything deployment needs
12. Push to GitHub

## Step 1: Install and import libraries
We need `xgboost` (a strong model for tabular data) on top of the libraries we already used in Phase 1.

In [ ]:
# Install xgboost (quiet mode: -q hides the long install messages)
!pip install -q xgboost

import os                      # work with folders and files
import json                    # save small info files (metrics, column names)
import time                    # measure how long each model takes to train
import joblib                  # save the trained model to a .pkl file
import numpy as np             # numbers and arrays
import pandas as pd            # tables (DataFrames)
import matplotlib.pyplot as plt  # charts

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from xgboost import XGBClassifier

# A fixed "seed" makes our results repeatable: anyone who runs this gets the same numbers
SEED = 42
print("Libraries ready")

## Step 2: Load the cleaned data from Phase 1
Phase 1 saved `train_clean.parquet` and `test_clean.parquet` in `data/processed/`.
- If you are running this in the **same Colab session** as Phase 1, the files are already there.
- If you opened a **fresh session**, the cell below downloads your GitHub repo so the files are available. Make sure the processed files were pushed to the repo, otherwise re-run the Phase 1 notebook first.

In [ ]:
# If the cleaned files are not in this session, get them from the group GitHub repo
if not os.path.exists("data/processed/train_clean.parquet"):
    !git clone https://github.com/Odewenu/network-anomaly-detection.git
    %cd network-anomaly-detection

# Load the cleaned train and test tables
train = pd.read_parquet("data/processed/train_clean.parquet")
test = pd.read_parquet("data/processed/test_clean.parquet")

print("Train shape:", train.shape)   # expected (96822, 66)
print("Test shape: ", test.shape)    # expected (48400, 66)

# "label" is what we want to predict (0 = normal, 1 = attack). Everything else is input.
X_full = train.drop(columns="label")   # inputs (features) for training
y_full = train["label"]                # answers (labels) for training
X_test = test.drop(columns="label")    # inputs for the final test
y_test = test["label"]                 # answers for the final test

print("Number of input features:", X_full.shape[1])   # expected 65

## Step 3: Split a validation set from the training data
**Why?** We need some data to compare and tune models on. If we used the test set for this, we would be "peeking" and our final score would look better than it really is.

So we cut 20% of the training data into a **validation set**.
- `X_train`, `y_train`: 80% of train, used to teach the models
- `X_val`, `y_val`: 20% of train, used to compare and tune models
- `X_test`, `y_test`: untouched until Step 8

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size=0.2,        # 20% goes to validation
    stratify=y_full,      # keep the normal/attack ratio the same in both parts
    random_state=SEED     # makes the split repeatable
)

print("Train part:     ", X_train.shape)
print("Validation part:", X_val.shape)
print("Test set:       ", X_test.shape)

# Check the class balance is similar everywhere (should be roughly 50/50)
print("\nAttack share in train part:     ", round(y_train.mean() * 100, 2), "%")
print("Attack share in validation part:", round(y_val.mean() * 100, 2), "%")
print("Attack share in test set:       ", round(y_test.mean() * 100, 2), "%")

## Step 4: Define how we measure a model
We use one helper function so every model is scored the same way.

| Metric | Plain meaning | Why it matters here |
|---|---|---|
| **Accuracy** | % of all predictions that are right | Easy to understand, but can hide problems |
| **Precision** | Of everything flagged as attack, how much really was an attack | Low precision = many false alarms |
| **Recall** | Of all real attacks, how many we caught | **Most important:** a missed attack is the costly mistake |
| **F1** | Balance of precision and recall | One number to rank models |
| **ROC-AUC / PR-AUC** | How well the model ranks attacks above normal traffic, at any threshold | Fair comparison that does not depend on a cut-off |

In [ ]:
def evaluate(model, X, y, name="model"):
    """Score a trained model on data X with true labels y. Returns a dictionary of metrics."""
    pred = model.predict(X)                        # hard predictions: 0 or 1
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X)[:, 1]       # probability of "attack"
    else:
        score = pred                               # fallback if the model gives no probability
    return {
        "model": name,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
        "roc_auc": roc_auc_score(y, score),
        "pr_auc": average_precision_score(y, score),
    }

print("Evaluation helper ready")

## Step 5: Train baseline models and compare them
A **baseline** is a quick first attempt with sensible settings. We train four different types so we can justify our final choice in the report:

1. **Logistic Regression**: simple linear model, the "minimum bar" to beat
2. **Decision Tree**: easy-to-explain rules
3. **Random Forest**: many trees voting together
4. **XGBoost**: trees that learn from each other's mistakes, usually the strongest on tabular data

In [ ]:
# Dictionary of models: name -> model with starting settings
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                             eval_metric="logloss", random_state=SEED, n_jobs=-1),
}

results = []                                        # we collect one row of metrics per model
for name, model in models.items():
    start = time.time()                             # start the stopwatch
    model.fit(X_train, y_train)                     # TRAIN the model on the 80% part
    row = evaluate(model, X_val, y_val, name)       # SCORE it on the validation part
    row["train_seconds"] = round(time.time() - start, 1)
    results.append(row)
    print("Done:", name)

baseline = pd.DataFrame(results).set_index("model").round(4)
baseline = baseline.sort_values("f1", ascending=False)    # best model on top
baseline

In [ ]:
# Visual comparison of the models (good figure for the report)
baseline[["precision", "recall", "f1"]].plot(kind="bar", figsize=(9, 5), ylim=(0.5, 1.0))
plt.title("Baseline models on the validation set")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("baseline_comparison.png", dpi=150)     # saved so you can paste it in the report
plt.show()

## Step 6: Try an unsupervised anomaly detector (Isolation Forest)
Our project is called *anomaly detection*, so we also try a model that never sees attack labels.

**Idea:** train it on **normal traffic only**. It learns what "normal" looks like and flags anything that looks unusual.

We expect it to score **lower** than the supervised models, because it never learns what attacks look like. That is a useful finding to discuss in the report.

In [ ]:
# Keep only the NORMAL rows (label 0) from the training part
X_normal_only = X_train[y_train == 0]

iso = IsolationForest(
    n_estimators=200,        # number of trees
    contamination=0.05,      # assumed share of anomalies in the data
    random_state=SEED,
    n_jobs=-1,
)
iso.fit(X_normal_only)       # learn what normal traffic looks like

# IsolationForest says -1 for anomaly and 1 for normal, so we convert to 1 (attack) / 0 (normal)
iso_pred = (iso.predict(X_val) == -1).astype(int)
iso_score = -iso.score_samples(X_val)     # higher value = more unusual

iso_result = {
    "model": "Isolation Forest (unsupervised)",
    "accuracy": accuracy_score(y_val, iso_pred),
    "precision": precision_score(y_val, iso_pred),
    "recall": recall_score(y_val, iso_pred),
    "f1": f1_score(y_val, iso_pred),
    "roc_auc": roc_auc_score(y_val, iso_score),
    "pr_auc": average_precision_score(y_val, iso_score),
}
pd.DataFrame([iso_result]).set_index("model").round(4)

## Step 7: Tune the best model
Tuning means searching for better settings (called **hyperparameters**). We use **RandomizedSearchCV**:
- It tries 20 random combinations from the ranges below.
- For each combination it does **3-fold cross-validation**: it trains on 2/3 of the training part and scores on the remaining 1/3, three times, and averages the result.
- It keeps the combination with the best **F1**.

> If Random Forest beat XGBoost in your Step 5 table, tell me and I will give you the same cell for Random Forest. The code below assumes XGBoost.

This cell can take a few minutes on Colab. That is normal.

In [ ]:
# The settings we want to search over
param_options = {
    "n_estimators": [200, 300, 500],        # number of trees
    "max_depth": [4, 6, 8, 10],             # how deep each tree can go
    "learning_rate": [0.03, 0.05, 0.1, 0.2],  # how fast it learns (smaller = slower but safer)
    "subsample": [0.7, 0.85, 1.0],          # share of rows each tree sees
    "colsample_bytree": [0.7, 0.85, 1.0],   # share of columns each tree sees
    "min_child_weight": [1, 3, 5],          # stops trees from fitting tiny groups (reduces overfitting)
}

search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1),
    param_distributions=param_options,
    n_iter=20,                                   # try 20 random combinations
    scoring="f1",                                # pick the best by F1
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)                     # run the search

print("Best settings:", search.best_params_)
print("Best cross-validation F1:", round(search.best_score_, 4))

# Score the tuned model on the validation part and compare with the untuned XGBoost
tuned_val = evaluate(search.best_estimator_, X_val, y_val, "XGBoost (tuned)")
pd.DataFrame([baseline.loc["XGBoost"].to_dict() | {"model": "XGBoost (baseline)"}, tuned_val]).set_index("model").round(4)

## Step 8: Final test on the test set (do this once)
Now we open the test set for the first time.

1. We retrain the tuned model on **all** the training data (train part + validation part), because more data usually helps.
2. We score it on the test set. These are the numbers you report.

**Do not go back and re-tune after seeing these results.** That would turn the test set into a second validation set.

> The test score will probably be lower than the validation score. UNSW-NB15's test set has a different mix of attacks than its training set, and in Phase 1 you also removed the train/test overlapping rows. So a drop is expected, and you should explain it in the report.

In [ ]:
# Build the final model with the best settings found in Step 7 and train on ALL training data
final_model = XGBClassifier(**search.best_params_, eval_metric="logloss", random_state=SEED, n_jobs=-1)
final_model.fit(X_full, y_full)

# Score on the untouched test set
final_metrics = evaluate(final_model, X_test, y_test, "XGBoost (final)")
print(pd.Series(final_metrics).round(4))

# Detailed per-class report
test_pred = final_model.predict(X_test)
print("\n", classification_report(y_test, test_pred, target_names=["Normal", "Attack"]))

In [ ]:
# Confusion matrix: rows = truth, columns = prediction
#   top-right    = false alarms (normal flagged as attack)
#   bottom-left  = MISSED attacks (the costly mistake)
ConfusionMatrixDisplay.from_predictions(y_test, test_pred,
                                        display_labels=["Normal", "Attack"], cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# ROC curve: the closer to the top-left corner, the better
RocCurveDisplay.from_estimator(final_model, X_test, y_test)
plt.title("ROC curve (test set)")
plt.savefig("roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
print(f"Normal correctly passed: {tn} | False alarms: {fp} | Missed attacks: {fn} | Attacks caught: {tp}")

## Step 9: Which attack types do we catch, and which do we miss?
In Phase 1 we saved `attack_cat` (the attack type) separately. We did not use it for training, but we can use it now to see how the model does on each attack type. This is strong material for the discussion section.

In [ ]:
# Load the attack type for each test row (same row order as the test file)
test_attack_cat = pd.read_parquet("data/processed/test_attack_cat.parquet")["attack_cat"].values

rows = []
for cat in sorted(set(test_attack_cat)):
    mask = test_attack_cat == cat                     # rows of this attack type
    if cat == "Normal":
        continue                                      # we only look at attacks here
    detected = (test_pred[mask] == 1).mean() * 100    # % of this attack type flagged as attack
    rows.append({"attack_type": cat, "samples": int(mask.sum()), "detected_percent": round(detected, 2)})

per_attack = pd.DataFrame(rows).sort_values("detected_percent")
per_attack.to_csv("per_attack_detection.csv", index=False)
per_attack

## Step 10: Which features matter most?
This shows which traffic features the model relies on. It helps explain *why* it works, which examiners like.

In [ ]:
importance = pd.Series(final_model.feature_importances_, index=X_full.columns)
top15 = importance.sort_values(ascending=False).head(15)

top15.iloc[::-1].plot(kind="barh", figsize=(8, 6), title="Top 15 most important features")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

top15.round(4)

## Step 11: Save the model and everything deployment needs
Phase 3 needs four things:
- the trained model (`xgb_model.pkl`)
- the **exact list of columns in the exact order** the model expects (`feature_columns.json`). The backend must build its input the same way.
- the scaler from Phase 1 (`data/processed/scaler.pkl`), already saved
- the metrics, for the UI and the report

In [ ]:
os.makedirs("models", exist_ok=True)

# 1. The trained model
joblib.dump(final_model, "models/xgb_model.pkl")

# 2. Column names and order
with open("models/feature_columns.json", "w") as f:
    json.dump(list(X_full.columns), f)

# 3. Final test metrics (convert numpy numbers to normal floats so JSON accepts them)
clean_metrics = {k: (v if isinstance(v, str) else float(v)) for k, v in final_metrics.items()}
with open("models/test_metrics.json", "w") as f:
    json.dump(clean_metrics, f, indent=2)

# 4. Comparison tables for the report
baseline.to_csv("models/baseline_comparison.csv")

# 5. Best settings, so the tuning can be reproduced
with open("models/best_params.json", "w") as f:
    json.dump({k: (float(v) if isinstance(v, float) else int(v)) for k, v in search.best_params_.items()}, f, indent=2)

# Show what was saved and the file sizes (the .pkl should be well under GitHub's 100 MB limit)
for file in sorted(os.listdir("models")):
    print(file, "-", round(os.path.getsize(os.path.join("models", file)) / 1e6, 3), "MB")

# Quick sanity check: reload the saved model and confirm it gives the same predictions
reloaded = joblib.load("models/xgb_model.pkl")
print("Reloaded model matches:", (reloaded.predict(X_test) == test_pred).all())

## Step 12: Push to GitHub
Run these in a terminal, or in a Colab cell with `!` in front. Replace the commit message if you like.

```
git add models/ notebooks/02_model_training.ipynb *.png per_attack_detection.csv
git commit -m "Phase 2: model training, tuned XGBoost, saved artifacts"
git push origin main
```

Put the notebook in the `notebooks/` folder, next to `01_data_cleaning_clean.ipynb`. If `git push` asks for credentials in Colab, use a GitHub personal access token as the password.

---
## What to take into the report (Part 2)
Keep these from this notebook:
- `baseline_comparison.png` and the baseline table: why XGBoost (or whichever won) was chosen
- the best settings from Step 7: your tuning approach
- `confusion_matrix.png`, `roc_curve.png` and the test metrics: your results
- `per_attack_detection.csv` and `feature_importance.png`: your discussion and analysis